# Visual Understanding Playground with VideoDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/scene-index/playground_scene_extraction.ipynb)

This notebook shows how to experiment with visual understanding prompts before turning results into a searchable index.

You will learn how to:

- Upload a video.
- Run `understand()` with the `vlm` analyzer using different segmentation strategies.
- Preview structured visual scene descriptions.
- Index the final visual understanding artifact.
- Search the resulting visual index and play matching moments.

This replaces the legacy scene extraction and scene-index workflow with the new understand, index, and retrieval pipeline.

## 1. Install Dependencies

In [ ]:
!pip install -q videodb python-dotenv


## 2. Connect to VideoDB

Enter your VideoDB API key when prompted.

Enter your VideoDB API key when prompted. You can get one from the [VideoDB Console](https://console.videodb.io). Get $20 free credits. **No credit card needed**.


In [2]:
import os
import time
from getpass import getpass
from uuid import uuid4

from IPython.display import display
from videodb import connect, play_stream

os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")

conn = connect()
coll = conn.get_collection()

print("Connected to VideoDB successfully.")

Please enter your VideoDB API Key: ··········
Connected to VideoDB successfully.


## 3. Upload a Video

We will use the same sample video for prompt experimentation and search.

In [3]:
video = coll.upload(url="https://www.youtube.com/watch?v=LejnTJL173Y")

print("Video ID:", video.id)
video.play()

Video ID: m-z-019f377a-091d-71c3-8a37-13715e23758e


## 4. Try a Time-Based Visual Understanding Prompt

Time-based segmentation is useful when you want consistent windows for comparing prompt behavior. Here, each record summarizes a 30-second segment.

In [4]:
time_prompt = (
    "Describe what is visually happening in this video segment. "
    "Mention the setting, people, actions, mood, and any important visual details."
)

time_understanding = video.understand(
    segmentation={
        "type": "time",
        "seconds": 30,
    },
    analyzers=[
        {
            "type": "vlm",
            "name": "time_scenes",
            "config": {
                "model": "pro",
                "prompt": time_prompt,
                "schema": {
                    "scene_description": "string",
                    "setting": "string",
                    "mood": "string",
                },
            },
        }
    ],
)

time_understanding.wait_until_complete(timeout=3600, poll_interval=15)
time_output = time_understanding.get_analyzer("time_scenes").get_output()

print("Time-based scenes:", len(time_output.get("scenes", [])))

Time-based scenes: 18


## 5. Preview Time-Based Results

Inspect a few records before deciding whether this prompt and segmentation work for your use case.

In [5]:
def preview_vlm_output(vlm_output, limit=5):
    for i, scene in enumerate(vlm_output.get("scenes", [])[:limit], 1):
        data = scene.get("data", {})
        print(f"Scene {i}: {scene.get('start')}s - {scene.get('end')}s")
        print("Description:", data.get("scene_description", ""))
        print("Setting:", data.get("setting", ""))
        print("Mood:", data.get("mood", ""))
        print("----")


preview_vlm_output(time_output)

Scene 1: 0.0s - 30.0s
Description: The segment opens on a close, mid-shot of a middle-aged man in a dark suit and tie seated in an office-like space behind venetian blinds; he leans forward and speaks with a serious, measured expression. The camera then cuts to wide and medium shots of an outdoor gathering under a canopy or tent. Rows of people — mostly women seated in the foreground and a mix of men and women standing at the back — face forward, some holding small books (likely hymnals or Bibles). Several attendees raise their hands, clap, or press a hand to their chest in an act of prayer or praise. The crowd is backlit by daylight, leaving the front rows softly out of focus while the standing figures near the tent edge are clearer. Important visual details include the contrast between the controlled indoor close-up and the open, communal outdoor scene, the visible hymnals/books, raised hands and clapping, and the rural tree line beyond the tent.
Setting: an indoor office or intervie

## 6. Try a Shot-Based Visual Understanding Prompt

Shot-based segmentation is useful when you want records to follow visual cuts and shot changes. Here we ask for more searchable, retrieval-oriented descriptions.

In [6]:
search_prompt = (
    "Create a concise visual search description for this shot. "
    "Include visible people, actions, objects, location, and notable concepts. "
    "Use concrete words that would help someone find this moment later."
)

shot_understanding = video.understand(
    segmentation={
        "type": "shot",
    },
    analyzers=[
        {
            "type": "vlm",
            "name": "visual_search",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 3,
            },
            "config": {
                "model": "pro",
                "prompt": search_prompt,
                "schema": {
                    "scene_description": "string",
                    "primary_action": "string",
                    "setting": "string",
                },
            },
        }
    ],
)

shot_understanding.wait_until_complete(timeout=3600, poll_interval=15)
visual_search_output = shot_understanding.get_analyzer("visual_search").get_output()

print("Shot-based scenes:", len(visual_search_output.get("scenes", [])))

Shot-based scenes: 73


## 7. Preview Shot-Based Results

This preview helps you confirm that the indexed fields will be useful for search.

In [7]:
preview_vlm_output(visual_search_output)

Scene 1: 0.0s - 11.678s
Description: medium close-up of a middle-aged bald man wearing a dark suit, light blue shirt and striped tie, seated in an office chair. Frontal head-and-shoulders shot with horizontal blinds and glass partitions behind him; looks like an interview or tense meeting, neutral indoor lighting, formal business setting.
Setting: modern office meeting room with horizontal blinds and glass partitions
Mood: 
----
Scene 2: 11.678s - 24.441s
Description: Crowd shot from inside tent: blurred seated women with white veils in the foreground, a row of standing adults and teens in the background (man in suit and sunglasses holding a book, woman in school uniform holding a binder). Visible actions include hand-raising, clapping, and attentive listening. Objects and details: canopy poles, grassy field, bare trees; overall appears like an outdoor religious service or community gathering.
Setting: Outdoor daytime under a large canopy/tent in a grassy field with leafless trees in t

## 8. Index the Final Visual Understanding Artifact

Once you like the prompt and segmentation, index the artifact for retrieval.

In [8]:
VISUAL_PLAYGROUND_INDEX_NAME = f"visual_playground_{uuid4().hex[:8]}"
INDEX_READY_STATUSES = {"ready", "done"}
INDEX_ACTIVE_STATUSES = {"building", "processing"}


def wait_until_index_ready(video, index, timeout=1800, poll_interval=10):
    deadline = time.time() + timeout
    latest_index = index

    while time.time() < deadline:
        status = getattr(latest_index, "status", None)

        if status in INDEX_READY_STATUSES:
            return latest_index

        if status and status not in INDEX_ACTIVE_STATUSES:
            raise RuntimeError(f"Index build ended with status: {status}")

        time.sleep(poll_interval)

        try:
            latest_index = video.get_index(index_id=index.index_id)
        except Exception as exc:
            if "not found" in str(exc).lower():
                continue
            raise

    raise TimeoutError(f"Index was not ready within {timeout} seconds.")


visual_index = video.index(
    name=VISUAL_PLAYGROUND_INDEX_NAME,
    source=visual_search_output,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["scene_description", "primary_action", "setting"],
        "text": ["scene_description"],
        "filter": ["primary_action", "setting"],
    },
)

visual_index = wait_until_index_ready(video, visual_index)

print("Visual playground index:", visual_index.index_id, visual_index.status)
print("Index fields:", visual_index.fields)

Visual playground index: ada4b404586445a0 ready
Index fields: {'filter': ['primary_action', 'setting'], 'semantic': ['scene_description', 'primary_action', 'setting'], 'text': ['scene_description']}


## 9. Search the Visual Index

Use `semantic_search()` for direct semantic retrieval over the fields configured as `semantic`.

In [9]:
search_results = video.semantic_search(
    query="people praying or singing in a gathering",
    index_names=[VISUAL_PLAYGROUND_INDEX_NAME],
    top_k=5,
    return_fields=["scene_description", "primary_action", "setting"],
)

for shot in search_results.shots:
    metadata = getattr(shot, "metadata", {}) or {}
    print(f"{shot.start:.2f}s - {shot.end:.2f}s")
    print(metadata.get("scene_description", ""))
    print("Primary action:", metadata.get("primary_action", ""))
    print("Setting:", metadata.get("setting", ""))
    print("----")

11.68s - 24.44s

Primary action: 
Setting: 
----
84.58s - 88.67s

Primary action: 
Setting: 
----
24.44s - 27.49s

Primary action: 
Setting: 
----
74.37s - 84.58s

Primary action: 
Setting: 
----
88.67s - 96.01s

Primary action: 
Setting: 
----


## 10. Play Search Results

Generate a stream from the matching timestamps and explicitly display the player.

In [11]:
result_timeline = [(shot.start, shot.end) for shot in search_results.shots]

if result_timeline:
    stream_link = video.generate_stream(result_timeline)
    player = play_stream(stream_link)
    display(player)
else:
    print("No matching visual moments found.")

## Conclusion

You used VideoDB's understanding pipeline as a visual-search playground.

The workflow is:

1. Try a segmentation strategy and VLM prompt with `understand()`.
2. Preview the artifact fields.
3. Index the artifact with `video.index()`.
4. Retrieve matching moments with `semantic_search()`.
5. Generate and display a stream from the result timestamps.

## Further Resources

- [Understanding Artifacts](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/understanding-artifacts)
- [Create an Index](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/create-an-index)
- [Search and Retrieval](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/search-and-retrieval/natural-language-query)

Related cookbook examples:

- [Custom Annotation Pipelines](custom_annotations.ipynb)
- [Advanced Visual Search](advanced_visual_search.ipynb)

If you have questions or feedback, reach out through [Discord](https://discord.gg/py9P639jGz) or [GitHub](https://github.com/video-db).
